In [ ]:
import gradio as gr
import requests
import numpy as np
from dotenv import load_dotenv

# Tải các biến môi trường từ file .env
load_dotenv()

OLLAMA_MODEL = "llava:7b"  # Chọn mô hình Ollama (có thể là llama, gpt-neo...)

# Hàm tính toán cosine similarity giữa hai vector
def cosine_similarity(vec1, vec2):
    """Tính cosine similarity giữa hai vector"""
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    
    # Tránh chia cho 0
    if norm_a == 0 or norm_b == 0:
        return 0
    
    return dot_product / (norm_a * norm_b)

# Hàm chính cho pipeline của demo
def ollama_vector_search(question):
    try:
        # Bước 1: Gửi yêu cầu POST tới Ollama API để sinh embedding
        response = requests.post("http://localhost:11434/api/generate", json={
            "model": OLLAMA_MODEL,
            "prompt": question,
            "stream": False
        })

        # Kiểm tra phản hồi từ Ollama
        if response.status_code == 200:
            embedding_response = response.json()
            query_embedding = embedding_response.get("embedding", [])
        else:
            return f"Error: {response.status_code} - {response.text}"

        if not query_embedding:
            return "Error: Failed to generate embedding for the question."

    except Exception as e:
        return f"Error generating embedding: {str(e)}"

    # Bước 2: Tìm kiếm trong Vector DB (Giả lập)
    # Các tài liệu mẫu sẽ được tạo ngẫu nhiên (thực tế, bạn sẽ sử dụng vector database thực tế)
    document_embeddings = [
        {"text": "Tài liệu về trí tuệ nhân tạo và các ứng dụng.", 
         "embedding": np.random.rand(len(query_embedding))},
        {"text": "Hướng dẫn sử dụng Ollama để chạy mô hình ngôn ngữ cục bộ.", 
         "embedding": np.random.rand(len(query_embedding))},
        {"text": "Vector search cho phép tìm kiếm ngữ nghĩa dựa trên sự tương đồng.",
         "embedding": np.random.rand(len(query_embedding))}
    ]

    best_doc = None
    best_similarity = -1  # Bắt đầu từ -1 vì cosine similarity nằm trong khoảng từ -1 đến 1

    # Tính toán độ tương đồng và tìm tài liệu phù hợp nhất
    for doc in document_embeddings:
        similarity = cosine_similarity(query_embedding, doc["embedding"])
        if similarity > best_similarity:
            best_doc = doc["text"]
            best_similarity = similarity

    if not best_doc:
        context = "Không tìm thấy tài liệu phù hợp."
    else:
        context = f"Tài liệu phù hợp (độ tương đồng: {best_similarity:.4f}): {best_doc}"

    # Bước 3: Sử dụng Ollama để sinh câu trả lời với ngữ cảnh
    try:
        response = requests.post("http://localhost:11434/api/generate", json={
            "model": OLLAMA_MODEL,
            "prompt": f"Câu hỏi: {question}\nNgữ cảnh: {context}",
            "stream": False
        })

        if response.status_code == 200:
            answer_response = response.json()
            answer = answer_response.get("response", "[No Answer]")
        else:
            return f"Error: {response.status_code} - {response.text}"

    except Exception as e:
        return f"Error generating answer: {str(e)}"

    return f"""Kết quả tìm kiếm:
{context}

Câu trả lời:
{answer}"""

# Giao diện Gradio
interface = gr.Interface(
    fn=ollama_vector_search,
    inputs=gr.Textbox(placeholder="Nhập câu hỏi của bạn ở đây..."),
    outputs=gr.Textbox(label="Kết quả"),
    title="Demo Ollama với Vector Search",
    description="Nhập câu hỏi và nhận câu trả lời sử dụng vector search và Ollama.",
    examples=[
        ["Trí tuệ nhân tạo là gì?"],
        ["Làm thế nào để sử dụng Ollama?"],
        ["Vector search hoạt động như thế nào?"]
    ]
)

if __name__ == "__main__":
    interface.launch(share=True)  # share=True để chia sẻ URL công khai tạm thời


* Running on local URL:  http://127.0.0.1:7884
* Running on public URL: https://2d1a963bcf7390b2ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
